In [2]:
# Importações e ínicio da sessão Spark

from pyspark.sql import SparkSession
import os
from pyspark.sql.functions import col, when
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.appName("MovieETL_Extract").getOrCreate()

print("Bibliotecas importadas e Spark iniciado")

Bibliotecas importadas e Spark iniciado


In [3]:
# Definindo pastas

base_path = "../input"
imdb_basics_path = os.path.join(base_path, "imdb/title.basics.tsv")
imdb_ratings_path = os.path.join(base_path, "imdb/title.ratings.tsv")
bronze_path = os.path.join(base_path, "bronze")

os.makedirs(bronze_path, exist_ok=True)

print("Pastas definidas e diretório bronze criado")

Pastas definidas e diretório bronze criado


In [4]:
# Definir a estrutura (Schema)

schema_basics = StructType([
    StructField("tconst", StringType(), True),
    StructField("titleType", StringType(), True),
    StructField("primaryTitle", StringType(), True),
    StructField("originalTitle", StringType(), True),
    StructField("isAdult", StringType(), True),
    StructField("startYear", StringType(), True),
    StructField("endYear", StringType(), True),
    StructField("runtimeMinutes", StringType(), True),
    StructField("genres", StringType(), True)
])

print("Esquema definido")

Esquema definido


In [5]:
# Ler o IMDB Basics

basics_df = (
    spark.read.csv(
        imdb_basics_path,
        sep="\t",
        header=True,
        schema=schema_basics,
        encoding="UTF-8"
    )
    .filter(col("titleType") == "movie")
    .withColumn("startYear", when(col("startYear") == "\\N", None).otherwise(col("startYear")).cast("int"))
    .filter(col("startYear") >= 2000)
)

basics_df.write.mode("overwrite").parquet(os.path.join(bronze_path, "basics.parquet"))
print(f"Extraidos {basics_df.count()} filmes basics para bronze.")

Extraidos 363787 filmes basics para bronze.


In [6]:
# Ler o IMDB RATINGS

ratings_df = spark.read.csv(imdb_ratings_path, sep="\t", header=True, inferSchema=True)

ratings_df.write.mode("overwrite").parquet(os.path.join(bronze_path, "ratings.parquet"))
print(f"Extraido {ratings_df.count()} ratings para bronze.")

Extraido 1627391 ratings para bronze.


In [7]:
# Para o Spark

spark.stop()